<!--
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

<style>
  .nav-link {
    color: #1a73e8;
    text-decoration: none;
    font-weight: 500;
    margin: 0 2px;
  }
  .nav-link:hover {
    text-decoration: underline;
  }
  .nav-strong {
    color: #202124;
    font-weight: 600;
    margin: 0 2px;
  }
</style>
<div style='margin-bottom: 24px; font-size: 13px; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; text-align: center; border-bottom: 1px solid #e0e0e0; padding-bottom: 12px; line-height: 1.8;'>
  <a class='nav-link' data-page='index' href='index.html'>Home</a> |
  <a class='nav-link' data-page='extending_modules_and_functions_via_composition' href='extending_modules_and_functions_via_composition.html'>Composition</a> |
  <span class='nav-strong'>Custom Autograd</span> |
  <a class='nav-link' data-page='background_on_aten_ops' href='background_on_aten_ops.html'>ATen Ops</a> |
  <a class='nav-link' data-page='background_on_torch_compile' href='background_on_torch_compile.html'>torch.compile</a> |
  <a class='nav-link' data-page='background_on_torch_compile_on_tpus_via_xla' href='background_on_torch_compile_on_tpus_via_xla.html'>Compile on TPU</a> |
  <a class='nav-link' data-page='quantized_sum' href='quantized_sum.html'>JAX Custom Ops</a> |
  <a class='nav-link' data-page='quant' href='quant.html'>Python Baseline</a> |
  <a class='nav-link' data-page='custom_ops_via_pallas' href='custom_ops_via_pallas.html'>Pallas TPU Kernel</a>
</div>

# torch.autograd.Function

`torch.autograd.Function` is the next approach to extending the functionality of
PyTorch. It enables developers to define a custom backward pass.

As a toy problem, suppose you decide that you want to modify the vanishing
gradients problem of sigmoid and instead clamp the gradient to a minimum of
0.01.

Look at the sample implementation below. Notice that:

*   `torch.autograd.Function` is subclassed
*   `forward` and `backward` are overridden
*   The `backward` implements the key clamping logic using the PyTorch functions
    `torch.where` and the Python operators `<=` and `>`.
*   The output of the forward pass is saved for the backward pass.

As a manual test, the main function prints a table of the outputs and grads for
both the custom sigmoid and the regular `torch.sigmoid`.

Input  | Custom Output | Regular Output | Custom Grad | Regular Grad
-----: | ------------: | -------------: | ----------: | -----------:
-100.0 | 0.0000        | 0.0000         | 0.0100      | 0.0000
-10.0  | 0.0000        | 0.0000         | 0.0100      | 0.0000
-1.0   | 0.2689        | 0.2689         | 0.2500      | 0.1966
0.0    | 0.5000        | 0.5000         | 0.2500      | 0.2500
1.0    | 0.7311        | 0.7311         | 0.2500      | 0.1966
10.0   | 1.0000        | 1.0000         | 0.0100      | 0.0000
100.0  | 1.0000        | 1.0000         | 0.0100      | 0.0000

> **Warning**: In PyTorch 1.0, custom ops (a topic discussed below) were implemented
> using this same mechanism. In PyTorch 2.0, custom ops have their own first
> class support and should not use `torch.autograd.Function`.

In [1]:
from absl import app as absl_app
import torch

# Uncomment this line in a future exercise.
torch._logging.set_logs(aot_graphs=True)  # pylint: disable=protected-access

Device type: tpu, Device index: default
Initializing TPU distributed runtime


I0000 00:00:1787340688.313176   78604 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/libtpu/libtpu.so
I0000 00:00:1787340688.329094   78604 tpu_profiler_plugin.cc:791] PyTorch PrivateUse1ProfilerRegistry not found. Native TPU profiling will be disabled (this is expected on PyTorch < 2.12).
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
I0000 00:00:1787340688.350345   78604 device_rt.cc:130] PjRt runtime initialization deferred for tpu


In [2]:
class SigmoidNoVanishing(torch.autograd.Function):
  """Custom Sigmoid function to avoid vanishing gradients.

  The gradient is forced to be either 0.01 or 1.0.
  """

  @staticmethod
  def forward(ctx, x):
    result = torch.sigmoid(x)
    ctx.save_for_backward(result)
    return result

  @staticmethod
  def backward(ctx, grad_output):
    (result,) = ctx.saved_tensors
    local_grad = torch.where(
        torch.logical_and(result > 0.1, result <= 0.9),
        torch.tensor(0.25, device=result.device),
        torch.tensor(0.01, device=result.device),
    )
    return grad_output * local_grad

In [3]:
def main():
  # Create variant of sigmoid:
  sigmoid_no_vanishing = SigmoidNoVanishing.apply
  sigmoid_no_vanishing = torch.compile(
      sigmoid_no_vanishing, backend="aot_eager"
  )

  # Input data.
  x = torch.tensor(
      [[-100.0], [-10.0], [-1.0], [0.0], [1.0], [10.0], [100.0]],
      requires_grad=True,
  )
  y = sigmoid_no_vanishing(x)

  # Reference sigmoid for comparison
  x_ref = x.detach().clone().requires_grad_(True)
  y_ref = torch.sigmoid(x_ref)

  # Verify outputs are identical
  _ = torch.allclose(y, y_ref)

  y.backward(torch.ones_like(y))
  y_ref.backward(torch.ones_like(y_ref))

  print(
      f"{'Input':>8} | {'Custom Output':>15} | {'Regular Output':>15} |"
      f" {'Custom Grad':>12} | {'Regular Grad':>12}"
  )
  print("-" * 74)
  for i in range(len(x)):
    print(
        f"{x[i].item():>8.1f} | {y[i].item():>15.4f} |"
        f" {y_ref[i].item():>15.4f} | {x.grad[i].item():>12.4f} |"
        f" {x_ref.grad[i].item():>12.4f}"
    )

main()

I0000 00:00:1787340688.375489   78604 pjrt_api.cc:168] The PJRT plugin has PJRT API version 0.114. The framework PJRT API version is 0.114.
E0821 19:31:28.412679   78604 hugepage_text.cc:344] RAW: File offset incorrectly aligned for file-backed THP: (6159b9400000 & ~ffffffffffe00000) = 0 != 104000 = (104000 & ~ffffffffffe00000)
I0000 00:00:1787340689.698188   78604 pjrt_c_api_client.cc:188] PjRtCApiClient created.
I0821 19:31:29.825000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1648] [0/0] [__aot_graphs] aot_config id: 0, fw_metadata=ViewAndMutationMeta(input_info=[InputAliasInfo(is_leaf=True, mutates_data=False, mutates_metadata=False, mutations_hidden_from_autograd=True, mutations_under_no_grad_or_inference_mode=False, mutation_inductor_storage_resize=False, mutates_storage_metadata=False, requires_grad=True, keep_input_mutations=True)], output_info=[OutputAliasInfo(output_type=<OutputType.non_alias: 1>, raw_type=<class 'torch._subclasses.functional_tensor.Fu

I0821 19:31:29.826000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1655] [0/0] [__aot_graphs] TRACED GRAPH
I0821 19:31:29.826000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1655] [0/0] [__aot_graphs]  ===== Forward graph 0 =====
I0821 19:31:29.826000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1655] [0/0] [__aot_graphs]  /home/mateusz/miniforge3/envs/tpu-experiment/lib/python3.12/site-packages/torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
I0821 19:31:29.826000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1655] [0/0] [__aot_graphs]     def forward(self, primals_1: "f32[7, 1][1, 1]cpu"):
I0821 19:31:29.826000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1655] [0/0] [__aot_graphs]         # File: /tmp/ipykernel_78604/2479064696.py:9 in forward, code: result = torch.sigmoid(x)
I0821 19:31:29.826000 78604 site-packages/torch/_functorch/_aot_autograd/graph

I0821 19:31:29.827000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1666] [0/0] [__aot_graphs] TRACED GRAPH
I0821 19:31:29.827000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1666] [0/0] [__aot_graphs]  ===== Backward graph 0 =====
I0821 19:31:29.827000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1666] [0/0] [__aot_graphs]  <eval_with_key>.6 class GraphModule(torch.nn.Module):
I0821 19:31:29.827000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1666] [0/0] [__aot_graphs]     def forward(self, logical_and: "b8[7, 1][1, 1]cpu", tangents_1: "f32[7, 1][1, 1]cpu"):
I0821 19:31:29.827000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1666] [0/0] [__aot_graphs]         # File: /tmp/ipykernel_78604/2479064696.py:18 in backward, code: torch.tensor(0.25, device=result.device),
I0821 19:31:29.827000 78604 site-packages/torch/_functorch/_aot_autograd/graph_compile.py:1666] [0/0] [__aot_gr

   Input |   Custom Output |  Regular Output |  Custom Grad | Regular Grad
--------------------------------------------------------------------------
  -100.0 |          0.0000 |          0.0000 |       0.0100 |       0.0000
   -10.0 |          0.0000 |          0.0000 |       0.0100 |       0.0000
    -1.0 |          0.2689 |          0.2689 |       0.2500 |       0.1966
     0.0 |          0.5000 |          0.5000 |       0.2500 |       0.2500
     1.0 |          0.7311 |          0.7311 |       0.2500 |       0.1966
    10.0 |          1.0000 |          1.0000 |       0.0100 |       0.0000
   100.0 |          1.0000 |          1.0000 |       0.0100 |       0.0000
